# FinGPT-Style Evaluation on Gold Commodity Dataset (Groq API Version)

This notebook mirrors the structure of your FinBERT notebook, but evaluates a **FinGPT-style sentiment workflow** on the Gold Commodity dataset using the **Groq API**.

It is set up so you can fill in your own key and model choice.


## 0. Installation

In [1]:

%pip install -U openai datasets scikit-learn pandas torch

  Using cached openai-2.30.0-py3-none-any.whl.metadata (29 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_

## Device Check

In [2]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")


=== nvidia-smi (if available) ===
Wed Apr  8 14:45:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 590.44.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:81:00.0 Off |                  N/A |
| 42%   30C    P8             31W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

## 1. Environment and Reproducibility Setup

In [3]:
import random

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paste your Groq key directly below.
# Do not share this notebook or upload it publicly after adding your real key.
GROQ_API_KEY = "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"

print("GROQ_API_KEY set:", GROQ_API_KEY != "your_groq_api_key_here" and bool(GROQ_API_KEY))


GROQ_API_KEY set: True


## 2. Dataset Loading

In [6]:
import io
import requests
import pandas as pd

LABEL_SET = {"negative", "neutral", "positive"}

FPB_URL = (
    "https://raw.githubusercontent.com/maxwellsarpong/"
    "NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt"
)

resp = requests.get(FPB_URL, timeout=60)
resp.raise_for_status()

gold_raw_df = pd.read_csv(
    io.StringIO(resp.text),
    sep="@",
    header=None,
    names=["sentence", "label_text"],
    engine="python",
    encoding="latin-1",
)

gold_raw_df["sentence"] = gold_raw_df["sentence"].astype(str).str.strip()
gold_raw_df["label_text"] = gold_raw_df["label_text"].astype(str).str.strip().str.lower()

gold_raw_df = gold_raw_df.dropna()
gold_raw_df = gold_raw_df[gold_raw_df["label_text"].isin(LABEL_SET)].reset_index(drop=True)

gold_eval_df = gold_raw_df[["sentence", "label_text"]].copy()

print(f"Gold eval samples: {len(gold_eval_df)}")
print(gold_eval_df["label_text"].value_counts())
print(gold_eval_df.head())

Gold eval samples: 3453
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64
                                            sentence label_text
0  According to Gran , the company has no plans t...    neutral
1  With the new production plant the company woul...   positive
2  For the last quarter of 2010 , Componenta 's n...   positive
3  In the third quarter of 2010 , net sales incre...   positive
4  Operating profit rose to EUR 13.1 mn from EUR ...   positive


## 3. FinGPT-Style Model Setup via Groq

This notebook uses a **prompted chat-completion approach** instead of a classifier head, so it behaves more like a FinGPT-style instruction workflow.

Swap `MODEL_NAME` to any Groq-hosted model you want to test.


In [7]:
import re
from openai import OpenAI

GROQ_API_KEY = "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"

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("Replace the placeholder in GROQ_API_KEY with your actual Groq API key before running.")

MODEL_NAME = "llama-3.1-8b-instant"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

print(f"Using Groq model: {MODEL_NAME}")

Using Groq model: llama-3.1-8b-instant


## 4. Prompt Template and Prediction Parser

In [8]:
LABELS = ["negative", "neutral", "positive"]

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier. "
    "Classify the sentiment of the given financial or commodity-market text. "
    "Return only one lowercase label from: negative, neutral, positive."
)

def build_prompt(text: str) -> str:
    return f"""Determine the sentiment of this financial text.

Text: {text}

Return only one label from: negative, neutral, positive."""

def parse_label(output_text: str) -> str:
    cleaned = output_text.strip().lower()

    for label in ["negative", "neutral", "positive"]:
        if re.search(rf'\b{label}\b', cleaned):
            return label

    if "bearish" in cleaned or "loss" in cleaned or "drop" in cleaned:
        return "negative"
    if "bullish" in cleaned or "gain" in cleaned or "rise" in cleaned:
        return "positive"

    return "neutral"


## 5. Batched Inference

In [9]:
def groq_predict_one(text: str, temperature: float = 0) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(text)},
        ],
    )
    raw = response.choices[0].message.content.strip()
    return parse_label(raw)

def run_fingpt_batched(sentences, batch_size=8, temperature=0):
    # Groq chat completions are called one prompt at a time here.
    # batch_size is kept only to mirror the structure of your earlier notebook.
    preds = []
    total = len(sentences)

    for i, text in enumerate(sentences, start=1):
        pred = groq_predict_one(text, temperature=temperature)
        preds.append(pred)

        if i % batch_size == 0 or i == total:
            print(f"Processed {i}/{total}")

    return preds


## 6. Full Evaluation Metrics

In [11]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, matthews_corrcoef
import pandas as pd

# keep only first 200 rows
gold_eval_df = gold_raw_df.head(200).copy()

true_labels = gold_eval_df["label_text"].tolist()
pred_labels = run_fingpt_batched(
    gold_eval_df["sentence"].tolist(),
    batch_size=20,
    temperature=0
)

metrics = {
    "macro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0),
    "micro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="micro", zero_division=0),
    "weighted_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0),
    "accuracy": accuracy_score(true_labels, pred_labels),
    "mcc": matthews_corrcoef(true_labels, pred_labels),
}

metrics_df = pd.DataFrame([metrics])
print(metrics_df)

print("\nClassification Report:\n")
print(classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0))

Processed 20/200
Processed 40/200
Processed 60/200
Processed 80/200
Processed 100/200
Processed 120/200
Processed 140/200
Processed 160/200
Processed 180/200
Processed 200/200
   macro_f1  micro_f1  weighted_f1  accuracy       mcc
0  0.329485      0.64     0.739033      0.64  0.275271

Classification Report:

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         0
     neutral       0.12      1.00      0.22        10
    positive       1.00      0.62      0.77       190

    accuracy                           0.64       200
   macro avg       0.38      0.54      0.33       200
weighted avg       0.96      0.64      0.74       200



## 7. Save Predictions

In [12]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/fingpt_style_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pred_df = gold_eval_df.copy()   # use the 200-row subset
pred_df["pred_label"] = pred_labels
pred_df["model_name"] = MODEL_NAME
pred_df["provider"] = "groq"

csv_path = EXPORT_DIR / "gold_fingpt_style_predictions_groq.csv"
pred_df.to_csv(csv_path, index=False)

print(f"Saved predictions to: {csv_path}")
print(pred_df.head())

Saved predictions to: experiments/sentiment_agent/fingpt_style_outputs/gold_fingpt_style_predictions_groq.csv
                                            sentence label_text pred_label  \
0  According to Gran , the company has no plans t...    neutral    neutral   
1  With the new production plant the company woul...   positive   positive   
2  For the last quarter of 2010 , Componenta 's n...   positive   positive   
3  In the third quarter of 2010 , net sales incre...   positive   positive   
4  Operating profit rose to EUR 13.1 mn from EUR ...   positive    neutral   

             model_name provider  
0  llama-3.1-8b-instant     groq  
1  llama-3.1-8b-instant     groq  
2  llama-3.1-8b-instant     groq  
3  llama-3.1-8b-instant     groq  
4  llama-3.1-8b-instant     groq  


## 8. Optional Save of Run Metadata

Since this notebook uses API inference, there is no local model/tokenizer to save.
Instead, this cell saves the run configuration for reproducibility.


In [13]:
from pathlib import Path
import json

MODEL_EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/fingpt_style_groq_run")
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run_config = {
    "provider": "groq",
    "model_name": MODEL_NAME,
    "seed": SEED,
    "dataset_name": "Financial PhraseBank 75Agree",
    "dataset_source": "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt",
    "labels": LABELS,
    "num_samples": len(gold_eval_df),
    "columns": list(gold_eval_df.columns),
}

config_path = MODEL_EXPORT_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print(f"Saved run metadata to: {config_path}")
print(json.dumps(run_config, indent=2))

Saved run metadata to: experiments/sentiment_agent/saved_models/fingpt_style_groq_run/run_config.json
{
  "provider": "groq",
  "model_name": "llama-3.1-8b-instant",
  "seed": 42,
  "dataset_name": "Financial PhraseBank 75Agree",
  "dataset_source": "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt",
  "labels": [
    "negative",
    "neutral",
    "positive"
  ],
  "num_samples": 200,
  "columns": [
    "sentence",
    "label_text"
  ]
}


## 9. Cleanup

In [14]:
import gc

for name in ["client"]:
    if name in globals():
        del globals()[name]

gc.collect()

if "torch" in globals():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Cleanup complete.")

Cleanup complete.
